# 03. Stage 2 resilience measures and regularized predictive models

Combines the 2018 OMB Stage 1 outcome and pre-February-2020 predictor datasets. Predictive features use only 2018 ACS five-year and 2018–2019 Q2 QCEW information that was eligible for public availability as of February 2020. Post-March-2020 outcomes *are targets only*.

The four principal resilience measures are: (1) 2020 resistance shortfall, (2) recovery gain relative to the 2020 trough, (3) late-2023 persistence shortfall, and (4) cumulative positive employment deficit as percent-point-month area. Shortfall metrics are *higher = worse*, recovery gain is *higher = better*. A qualifying **recovery episode** requires three consecutive months at or above the counterfactual **after the 2020 trough**, and only applies if the trough gap was negative. The start and confirmation month are distinguished. An unrecovered eligible region is right-censored; never-below regions have recovery status NA (not 'failure'). Recovery does not imply persistence through December 2023.

**Full-run configuration:** All eligible regions are required; any partial Stage 1 input is rejected. Historical stress outcomes from notebook 01 are incorporated. Exploratory model failures are explicitly logged rather than silently stopping the preparation of the full nested geographic analysis in notebook 04.


Primary design measures added: Apr--Jun peak resistance, fraction of the initial peak deficit closed at months 12/24/46, and point-in-time persistence at those months. Recovery fractions can exceed 1 or be negative; they are missing for zero initial peak. Prior average-loss and gain measures remain sensitivity columns.


In [1]:
from pathlib import Path
import os,json,hashlib,warnings
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold,GroupShuffleSplit,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge,Lasso,ElasticNet
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
ROOT=Path(os.environ['RL_PROJECT_ROOT']).resolve() if os.environ.get('RL_PROJECT_ROOT') else next(
 (p for p in [Path.cwd(),*Path.cwd().parents] if (p/'output'/'stage1_omb2018_manifest.json').exists()),None)
if ROOT is None:raise FileNotFoundError('Run notebooks 00–02 first')
PROC=ROOT/'data'/'processed';OUT=ROOT/'output'
GEO=json.loads((PROC/'omb2018_geography_manifest.json').read_text())
STAGE1=json.loads((OUT/'stage1_omb2018_manifest.json').read_text())
PREDMETA=json.loads((OUT/'stage2_omb2018_predictors_manifest.json').read_text())
if {GEO['geography_vintage'],STAGE1['geography_vintage'],PREDMETA['geography_vintage']}!={'2018-09'}:
 raise ValueError('Cross-vintage merge prohibited')
if STAGE1.get('design_schema')!='released_by_20200229_v2' or PREDMETA.get('design_schema')!='released_by_20200229_v2':
 raise ValueError('Rerun revised 01 and 02; old-design inputs cannot be mixed with revised outcomes')
if not STAGE1.get('all_regions_included',False):
 raise ValueError('Stage 1 is a partial RL_MAX_CBSAS run; not valid for national Stage 2. Full-run notebooks do not accept partial input.')
if STAGE1['model_training_cutoff']!='2019-06-01':raise ValueError('Stage 1 uses post-cutoff training')
if PREDMETA['information_cutoff']!='2020-02-29':raise ValueError('Predictors use incorrect cutoff')
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for x in iter(lambda:f.read(4*1024*1024),b''):h.update(x)
 return h.hexdigest()
if sha256(OUT/'stage2_omb2018_extended_predictors_manifest.json')!=PREDMETA['extended_predictors_manifest_sha256']:
 raise ValueError('Extended predictor provenance changed; rerun 02')
DATA=PROC/'stage1_omb2018_observed_vs_counterfactual_2020_2023.csv'
PRED=PROC/'stage2_omb2018_pre2020_predictors.csv'
if sha256(DATA)!=STAGE1['outcomes_sha256'] or sha256(PRED)!=PREDMETA['predictors_sha256']:
 raise ValueError('Stage inputs changed since their provenance manifests')

## 1. Resilience outcomes from the 46 post-March-2020 months

In [2]:
gaps=pd.read_csv(DATA,dtype={'cbsa_code':'string'},parse_dates=['date'])
gaps.cbsa_code=gaps.cbsa_code.str.zfill(5)
if gaps.duplicated(['cbsa_code','date']).any():raise ValueError('Duplicate month rows')
if not gaps.groupby('cbsa_code').size().eq(46).all():raise ValueError('Incomplete 46-month evaluation')
if gaps.date.min()!=pd.Timestamp('2020-03-01') or gaps.date.max()!=pd.Timestamp('2023-12-01'):
 raise ValueError('Unexpected evaluation dates')
if (gaps.counterfactual_employment<=0).any():raise ValueError('Nonpositive counterfactual employment')
gaps['gap_pct']=100*(gaps.employment/gaps.counterfactual_employment-1)
gaps['shortfall_pct']=(-gaps.gap_pct).clip(lower=0)
gaps.to_csv(PROC/'stage2_omb2018_monthly_gaps.csv',index=False)

def measure(s):
 s=s.sort_values('date').reset_index(drop=True)
 q=lambda start,end:s.loc[s.date.between(start,end),'gap_pct']
 first=q('2020-04-01','2020-06-01')
 trough=s.loc[s.date.between('2020-04-01','2020-12-01'),'gap_pct'].min()
 last=q('2023-10-01','2023-12-01')
 # Start a recovery search only after the observed 2020 trough.
 eligible=(trough<0)
 trough_ix=int(s.loc[s.date.between('2020-04-01','2020-12-01'),'gap_pct'].idxmin())
 above=s.gap_pct.ge(0).to_numpy()
 recovery_start=next((i for i in range(trough_ix+1,len(above)-2)
                      if above[i:i+3].all()),None) if eligible else None
 recovery_confirm=recovery_start+2 if recovery_start is not None else None
 return pd.Series({
  'resistance_shortfall_pct_aprjun2020':float((-first).clip(lower=0).mean()),
  'initial_2020_trough_gap_pct':float(trough),
  'recovery_gain_from_trough_pp':float(last.mean()-trough),
  'persistence_shortfall_pct_octdec2023':float((-last).clip(lower=0).mean()),
  'cumulative_shortfall_12_pp_months':float(s.shortfall_pct.iloc[:12].sum()),
  'cumulative_shortfall_24_pp_months':float(s.shortfall_pct.iloc[:24].sum()),
  'cumulative_shortfall_46_pp_months':float(s.shortfall_pct.sum()),
  'recovery_eligible_below_counterfactual':int(eligible),
  'trough_date_2020':s.date.iloc[trough_ix].strftime('%Y-%m-%d'),
  'recovered_by_dec2023':int(recovery_start is not None) if eligible else np.nan,
  'recovery_episode_start_date':s.date.iloc[recovery_start].strftime('%Y-%m-%d') if recovery_start is not None else None,
  'recovery_episode_confirm_date':s.date.iloc[recovery_confirm].strftime('%Y-%m-%d') if recovery_confirm is not None else None,
  'months_from_trough_to_recovery_start':float(recovery_start-trough_ix) if recovery_start is not None else np.nan,
  'months_to_sustained_recovery':float(recovery_confirm-trough_ix) if recovery_confirm is not None else np.nan,
  'gap_pct_dec2023':float(s.gap_pct.iloc[-1]),
  'employment_2020apr':float(s.loc[s.date.eq(pd.Timestamp('2020-04-01')),'employment'].iloc[0]),
  **{f'cumulative_shortfall_{h}_pp_months_{scenario}':float(s[f'shortfall_pct_{scenario}'].iloc[:h].sum())
     for scenario in ('ensemble','damped') for h in (12,24,46)}})
rows=[]
for cbsa,s in gaps.groupby('cbsa_code',sort=True):
 r=measure(s);r['cbsa_code']=cbsa;rows.append(r)
outcomes=pd.DataFrame(rows)
import sys
if str(ROOT/'code') not in sys.path: sys.path.insert(0,str(ROOT/'code'))
from design_models import planned_outcomes
planned=[]
for cbsa,s in gaps.groupby('cbsa_code',sort=True):
    r=planned_outcomes(s);r['cbsa_code']=cbsa;planned.append(r)
outcomes=outcomes.merge(pd.DataFrame(planned),on='cbsa_code',validate='one_to_one')
for scenario in ('common_ets','short_window'):
    col=f'shortfall_pct_{scenario}'
    if col not in gaps: raise ValueError('Revised Stage 1 sensitivities absent; rerun 01')
    # Missing forecast sensitivity stays missing; never silently sum partial paths.
    totals=gaps.groupby('cbsa_code')[col].agg(lambda s:s.sum() if s.notna().sum()==46 else np.nan)
    outcomes[f'cumulative_shortfall_46_pp_months_{scenario}']=outcomes.cbsa_code.map(totals)
cols=['cbsa_code']+[x for x in outcomes if x!='cbsa_code']
outcomes=outcomes[cols]
# Optional scenario outcomes were built using pre-cutoff-only historical pseudo-origin
# fits. These are stress tests, not frequentist posterior/confidence intervals.
STRESS_SOURCE=PROC/'stage1_omb2018_historical_stress_outcomes.csv'
STRESS_USED=False
if STAGE1.get('historical_stress_paths_completed',False):
    if not STRESS_SOURCE.exists() or sha256(STRESS_SOURCE)!=STAGE1.get('historical_stress_outcomes_sha256'):
        raise ValueError('Historical stress outcomes are missing or changed since Stage 1 manifest')
    stress=pd.read_csv(STRESS_SOURCE,dtype={'cbsa_code':'string'})
    stress['cbsa_code']=stress.cbsa_code.str.zfill(5)
    if stress.duplicated(['cbsa_code','stress_origin']).any():
        raise ValueError('Duplicate historical stress scenario for CBSA')
    origins=STAGE1['historical_stress_origins']
    if len(stress)!=len(outcomes)*len(origins) or set(stress.cbsa_code)!=set(outcomes.cbsa_code):
        raise ValueError('Incomplete CBSA coverage in historical stress outcomes')
    for origin in origins:
        one=stress.loc[stress.stress_origin.eq(origin),
                       ['cbsa_code','cumulative_shortfall_46_pp_months']].copy()
        one=one.rename(columns={'cumulative_shortfall_46_pp_months':
                                f'cumulative_shortfall_46_pp_months_stress_{origin}'})
        outcomes=outcomes.merge(one,on='cbsa_code',how='left',validate='one_to_one')
        if outcomes[f'cumulative_shortfall_46_pp_months_stress_{origin}'].isna().any():
            raise ValueError(f'Incomplete stress scenario: {origin}')
    STRESS_USED=True
    print('Historical counterfactual stress outcomes attached:',origins)
outcomes.to_csv(PROC/'stage2_omb2018_resilience_outcomes.csv',index=False)
print('CBSA-level outcomes:',len(outcomes))

Historical counterfactual stress outcomes attached: ['2013_06', '2014_06', '2014_12']
CBSA-level outcomes: 936


## 2. Join predictors with strict vintage and sample validation

In [3]:
pred=pd.read_csv(PRED,dtype={'cbsa_code':'string'})
pred['cbsa_code']=pred.cbsa_code.str.zfill(5)
if pred.cbsa_code.duplicated().any():raise ValueError('Duplicate predictors')
merged=outcomes.merge(pred,on='cbsa_code',how='inner',validate='one_to_one')
if len(merged)!=len(outcomes):raise ValueError('Missing regional predictors in Stage 2')
if merged.geographic_group.isna().any():raise ValueError('Missing geographic validation group')
merged.to_csv(PROC/'stage2_omb2018_modeling_table.csv',index=False)
print('Matched regions:',len(merged),'Geographic groups:',merged.geographic_group.nunique())

Matched regions: 936 Geographic groups: 10


## 3. Ridge, Lasso, Elastic Net with geographically held-out validation

This is **exploratory only**: a 20% held-out *Census-division group* split is used for diagnostics; **the full nested geographic evaluation of all principal outcomes is in notebook 04**. inner GroupKFold tunes hyperparameters using training groups only. Imputation and scaling occur inside CV folds. Feature selection by availability and nonzero variance is fitted **within each inner training fold** using a Pipeline transformer (and again on the outer training set before testing). Models are **predictive associations**, not causal effects.

In [4]:
print('Revised principal Elastic Net/OLS/Random Forest/gradient-boosting comparisons run in notebook 04; prior single-split diagnostics are superseded.')


Revised principal Elastic Net/OLS/Random Forest/gradient-boosting comparisons run in notebook 04; prior single-split diagnostics are superseded.


## 4. Reproducibility and limitations

In [5]:
manifest3={'design_schema':'released_by_20200229_v2','geography_vintage':'2018-09','information_cutoff':'2020-02-29',
 'forecast_training_cutoff':'2019-06-01',
 'outcome_window':'2020-03 through 2023-12',
 'modeling_cbsa_count':len(merged),
 'predictors_sha256':sha256(PRED),'outcomes_sha256':sha256(DATA),
  'monthly_gaps_file':'data/processed/stage2_omb2018_monthly_gaps.csv',
  'monthly_gaps_sha256':sha256(PROC/'stage2_omb2018_monthly_gaps.csv'),
  'resilience_outcomes_file':'data/processed/stage2_omb2018_resilience_outcomes.csv',
  'resilience_outcomes_sha256':sha256(PROC/'stage2_omb2018_resilience_outcomes.csv'),
  'modeling_table_sha256':sha256(PROC/'stage2_omb2018_modeling_table.csv'),
  'historical_stress_outcomes_sha256':sha256(STRESS_SOURCE) if STRESS_USED else None,
  'historical_stress_origins':STAGE1['historical_stress_origins'] if STRESS_USED else [],
  'stage1_complete':bool(STAGE1.get('all_regions_included',False)),
  'primary_recovery_definition':'(initial Apr-Jun peak positive deficit + signed fixed-horizon gap) / peak; unbounded; undefined for zero peak',
 'resistance_definition':'Maximum Apr-Jun 2020 positive shortfall',
 'persistence_definition':'Positive deficit at months 12,24,46 since March 2020',
 'recovery_definition':'First 3 consecutive nonnegative gap months after negative 2020 trough; timing measured from trough to confirmation month',
 'limitations':[
  'Later downloaded QCEW archives may include revisions not published by February 2020.',
  'Counterfactuals are extrapolations, not identified causal COVID effects.',
  'Absent structural-zero QCEW sectors are zero; explicitly suppressed N remains missing, with fold-only imputation.',
  'Recovery duration is right-censored when not achieved by December 2023.',
  '2018 OMB delineations held constant across all years.']}
(OUT/'stage2_omb2018_modeling_manifest.json').write_text(json.dumps(manifest3,indent=2))
print('Stage 2 complete. Inspect coverage, data-vintage, model and geographic audits.')

Stage 2 complete. Inspect coverage, data-vintage, model and geographic audits.
